# Geovisor - Hospitales y Clinicas ITT

**Proyecto:** Gobierno de Datos 2026

**Modulo:** Espacializacion de Instituciones Prestadoras de Salud

**Repositorio:** [GitHub](https://github.com/j0rg3c45/Hospitales_Cl-nicas_ITT.git)

---

## Objetivo

Generar un visor geografico interactivo (Geovisor) que permita visualizar, filtrar y analizar individualmente cada hospital y clinica del territorio, consumiendo servicios de imagenes satelitales de Google como mapa base.

---
## 1. Instalacion de Dependencias

In [ ]:
%pip install folium --quiet

print("Dependencias instaladas.")

---
## 2. Importacion de Librerias

In [ ]:
import json
import os
import folium
from folium import FeatureGroup, LayerControl
from folium.plugins import MiniMap, Fullscreen, LocateControl

print("Librerias importadas.")
print(f"  Folium: {folium.__version__}")

---
## 3. Carga de Datos Geograficos

In [ ]:
# Opcion A: clonar el repositorio para tener el archivo disponible
if not os.path.exists("Hospitales_Cl-nicas_ITT"):
    !git clone https://github.com/j0rg3c45/Hospitales_Cl-nicas_ITT.git

DATA_PATH = "Hospitales_Cl-nicas_ITT/Data/hospitales_clinicas.geojson"

# Opcion B: subir manualmente si no se clono
if not os.path.exists(DATA_PATH):
    from google.colab import files
    print("Archivo no encontrado. Suba el archivo GeoJSON:")
    uploaded = files.upload()
    DATA_PATH = list(uploaded.keys())[0]

# Lectura del GeoJSON con json estandar
with open(DATA_PATH, 'r', encoding='utf-8') as f:
    geojson_data = json.load(f)

features = geojson_data['features']

print(f"Datos cargados.")
print(f"  Registros: {len(features)}")
print(f"  Campos: {list(features[0]['properties'].keys())}")

---
## 4. Exploracion de Datos

In [ ]:
# Mostrar resumen de cada institucion
print(f"{'#':<3} {'Nombre':<45} {'Lat':<12} {'Lon':<12}")
print("-" * 72)

for i, feat in enumerate(features):
    props = feat['properties']
    coords = feat['geometry']['coordinates']
    lon, lat = coords[0], coords[1]
    print(f"{i+1:<3} {props['nombre']:<45} {lat:<12.6f} {lon:<12.6f}")

---
## 5. Espacializacion / Geovisor

Seccion principal. Se construye el mapa interactivo con:

- Mapa base: Google Satellite, Google Hybrid, Google Maps, OpenStreetMap.
- Capas individuales: cada hospital/clinica como capa independiente.
- Layer Control: permite encender/apagar cada institucion.
- Popups informativos con datos de cada IPS.

In [ ]:
# --- Tiles de Google ---
GOOGLE_SATELLITE = "https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}"
GOOGLE_MAPS = "https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}"
GOOGLE_HYBRID = "https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}"

# Calcular centro del mapa
lats = [f['geometry']['coordinates'][1] for f in features]
lons = [f['geometry']['coordinates'][0] for f in features]
center_lat = sum(lats) / len(lats)
center_lon = sum(lons) / len(lons)

# Crear mapa base
mapa = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=13,
    tiles=None,
    control_scale=True
)

# Agregar basemaps
folium.TileLayer(
    tiles=GOOGLE_SATELLITE,
    attr="Google Satellite",
    name="Google Satellite",
    overlay=False
).add_to(mapa)

folium.TileLayer(
    tiles=GOOGLE_HYBRID,
    attr="Google Hybrid",
    name="Google Hybrid",
    overlay=False
).add_to(mapa)

folium.TileLayer(
    tiles=GOOGLE_MAPS,
    attr="Google Maps",
    name="Google Maps",
    overlay=False
).add_to(mapa)

folium.TileLayer(
    tiles="OpenStreetMap",
    name="OpenStreetMap",
    overlay=False
).add_to(mapa)

print(f"Mapa base creado. Centro: [{center_lat:.4f}, {center_lon:.4f}]")

In [ ]:
# --- Capas individuales por institucion ---

colores_iconos = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 'darkblue', 'darkgreen']

for idx, feat in enumerate(features):
    props = feat['properties']
    coords = feat['geometry']['coordinates']
    lon, lat = coords[0], coords[1]
    nombre = props.get('nombre', f'Institucion {idx+1}')
    color = colores_iconos[idx % len(colores_iconos)]

    # FeatureGroup individual para control de capas
    fg = FeatureGroup(name=nombre, show=True)

    # Popup con informacion
    popup_html = f"""
    <div style='font-family: Arial; font-size: 12px; min-width: 220px;'>
        <h4 style='margin-bottom: 5px;'>{nombre}</h4>
        <hr style='margin: 3px 0;'>
        <b>Direccion:</b> {props.get('direccion', 'N/A')}<br>
        <b>Comuna:</b> {props.get('comuna', 'N/A')}<br>
        <b>Municipio:</b> {props.get('municipio', 'N/A')}<br>
        <b>Departamento:</b> {props.get('departamento', 'N/A')}<br>
        <b>Lat:</b> {lat:.6f}<br>
        <b>Lon:</b> {lon:.6f}<br>
    </div>
    """

    # Marcador
    folium.Marker(
        location=[lat, lon],
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=nombre,
        icon=folium.Icon(color=color, icon='plus-sign', prefix='glyphicon')
    ).add_to(fg)

    fg.add_to(mapa)

print(f"{len(features)} instituciones agregadas como capas individuales.")

In [ ]:
# --- Controles del mapa ---

LayerControl(
    collapsed=False,
    position='topright'
).add_to(mapa)

MiniMap(
    toggle_display=True,
    position='bottomleft'
).add_to(mapa)

Fullscreen(
    position='topleft',
    title='Pantalla completa',
    title_cancel='Salir'
).add_to(mapa)

LocateControl(
    strings={'title': 'Mi ubicacion'}
).add_to(mapa)

print("Controles configurados: Layer Control, MiniMap, Fullscreen, Locate.")

---
## 6. Visualizacion del Geovisor

In [ ]:
# Renderizar mapa interactivo
mapa

---
## 7. Exportacion

In [ ]:
# Exportar como HTML interactivo
OUTPUT_PATH = "geovisor_hospitales_clinicas_ITT.html"

mapa.save(OUTPUT_PATH)
print(f"Mapa exportado: {OUTPUT_PATH}")

# Descargar en Colab
try:
    from google.colab import files
    files.download(OUTPUT_PATH)
    print("Descarga iniciada.")
except:
    print("Para descargar, abra el archivo desde el explorador de archivos.")

---
## 8. Conclusiones

### Resultados:
- Geovisor interactivo con mapa base de Google Satellite/Hybrid/Maps.
- 3 instituciones de salud representadas como capas individuales.
- Layer Control que permite encender/apagar cada hospital o clinica.
- Popups con direccion, comuna, municipio y coordenadas.

### Proximos pasos:
- Agregar mas instituciones al GeoJSON.
- Incorporar poligonos de cobertura y areas de influencia.
- Enriquecer popups con datos de servicios y capacidad instalada.
- Generar reportes automaticos por zona geografica.